3.0.1 Drive i putanje

Montira se Drive i postavljaju se putanje ka projektu, curated podacima i Runs folderu. Ovaj notebook upisuje rezultate u novi Runs/experiments_YYYYMMDD_HHMMSS folder.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, math, random
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Diplomski")
CURATED = ROOT / "Data" / "curated"
RUNS = ROOT / "Runs"
RUNS.mkdir(parents=True, exist_ok=True)

run_id = time.strftime("%Y%m%d_%H%M%S")
OUT_ROOT = RUNS / f"experiments_{run_id}"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("CURATED:", CURATED)
print("OUT_ROOT:", OUT_ROOT)

Mounted at /content/drive
ROOT: /content/drive/MyDrive/Diplomski
CURATED: /content/drive/MyDrive/Diplomski/Data/curated
OUT_ROOT: /content/drive/MyDrive/Diplomski/Runs/experiments_20260312_212038


3.0.2 Bootstrap helperi

Učitavaju se meta.json i split CSV fajlovi i standardizuju se kolone za label i putanju. Ovo obezbeđuje da isti pipeline radi za sva tri image dataseta i thyroid tabular.

In [2]:
import pandas as pd
import numpy as np

def load_meta(ds):
    with open(CURATED / ds / "meta.json", "r", encoding="utf-8") as f:
        return json.load(f)

def load_split(ds, split):
    return pd.read_csv(CURATED / ds / f"{split}.csv")

def infer_task(meta):
    t = meta.get("task_type")
    if t in ["image", "tabular"]:
        return t
    if "image" in json.dumps(meta).lower():
        return "image"
    return "tabular"

def label_col(meta, df):
    c = meta.get("label_col")
    if c and c in df.columns:
        return c
    for cand in ["label", "target", "y", "class", "Recurred"]:
        if cand in df.columns:
            return cand
    return df.columns[-1]

def path_col(meta, df):
    c = meta.get("path_col")
    if c and c in df.columns:
        return c
    for cand in ["path", "filepath", "image_path", "img_path", "file"]:
        if cand in df.columns:
            return cand
    return None

def save_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

3.0.3 Reproducibility i device

Postavlja se seed da bi eksperimenti bili ponovljivi i bira se GPU ako postoji. Ovde odmah vidiš da li radiš na cuda ili cpu.

In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms
from PIL import Image

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

device: cuda
NVIDIA L4


3.1 Image Dataset i transform preseti

Pravi se dataset koji čita slike iz CSV-a i mapira labele u indekse. Definišu se transform preseti za baseline i za jači tuning (bitno za SipakMed).

In [4]:
class ImageCsvDataset(Dataset):
    def __init__(self, df, pcol, ycol, label_to_idx=None, tfm=None):
        self.df = df.reset_index(drop=True)
        self.pcol = pcol
        self.ycol = ycol
        self.tfm = tfm

        labels = self.df[ycol].astype(str).tolist()
        if label_to_idx is None:
            uniq = sorted(list(set(labels)))
            self.label_to_idx = {u:i for i,u in enumerate(uniq)}
        else:
            self.label_to_idx = label_to_idx

        self.y = [self.label_to_idx[str(x)] for x in labels]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        p = str(self.df.iloc[i][self.pcol])
        y = self.y[i]
        im = Image.open(p).convert("RGB")
        if self.tfm is not None:
            im = self.tfm(im)
        return im, y

def make_tfm(preset, img_size):
    if preset == "baseline":
        return {
            "train": transforms.Compose([
                transforms.Resize((img_size, img_size)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.ToTensor(),
            ]),
            "eval": transforms.Compose([
                transforms.Resize((img_size, img_size)),
                transforms.ToTensor(),
            ])
        }
    if preset == "strong":
        return {
            "train": transforms.Compose([
                transforms.RandomResizedCrop(img_size, scale=(0.75, 1.0)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomRotation(10),
                transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.02),
                transforms.ToTensor(),
            ]),
            "eval": transforms.Compose([
                transforms.Resize((img_size, img_size)),
                transforms.ToTensor(),
            ])
        }
    raise ValueError("Unknown preset")

3.2 Loader + class weights

Loader pravi train/val/test DataLoader i vraća mapping labela. Class weights se računaju iz train skupa i opciono ulaze u loss.

In [5]:
def compute_class_weights(y_idx, num_classes):
    counts = np.bincount(np.array(y_idx), minlength=num_classes).astype(np.float32)
    w = counts.sum() / (counts + 1e-8)
    w = w / w.mean()
    return torch.tensor(w, dtype=torch.float32)

def make_image_loaders(ds, seed, batch_size=32, num_workers=2, img_size=224, tfm_preset="baseline"):
    set_seed(seed)
    meta = load_meta(ds)
    df_tr = load_split(ds, "train")
    df_va = load_split(ds, "val")
    df_te = load_split(ds, "test")

    ycol = label_col(meta, df_tr)
    pcol = path_col(meta, df_tr)

    tfm = make_tfm(tfm_preset, img_size)

    ds_tr = ImageCsvDataset(df_tr, pcol, ycol, label_to_idx=None, tfm=tfm["train"])
    ds_va = ImageCsvDataset(df_va, pcol, ycol, label_to_idx=ds_tr.label_to_idx, tfm=tfm["eval"])
    ds_te = ImageCsvDataset(df_te, pcol, ycol, label_to_idx=ds_tr.label_to_idx, tfm=tfm["eval"])

    dl_tr = DataLoader(ds_tr, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=torch.cuda.is_available())
    dl_va = DataLoader(ds_va, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=torch.cuda.is_available())
    dl_te = DataLoader(ds_te, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=torch.cuda.is_available())

    idx_to_label = {v:k for k,v in ds_tr.label_to_idx.items()}
    return dl_tr, dl_va, dl_te, ds_tr.label_to_idx, idx_to_label, ds_tr.y

3.3 Modeli: ResNet18 i EfficientNet-B0

Ovim dobijaš dva modela za poređenje uz isti pipeline. Na SipakMed obično EfficientNet-B0 ume da da mali boost.

In [6]:
def build_model(name, num_classes):
    if name == "resnet18":
        m = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        return m
    if name == "efficientnet_b0":
        m = torchvision.models.efficientnet_b0(weights=torchvision.models.EfficientNet_B0_Weights.DEFAULT)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
        return m
    raise ValueError("Unknown model")

3.4 Trening i evaluacija (sa 2-phase fine-tune opcijom)

Ovo trenira model, bira najbolji po val macro F1 i vraća test metrike i konfuzionu matricu. Može da radi samo head (freeze) ili kasnije delimično otključavanje (fine-tune).

In [7]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

@torch.no_grad()
def eval_loader(model, loader):
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        x = x.to(device)
        y = y.to(device) if torch.is_tensor(y) else torch.tensor(y).to(device)
        logits = model(x)
        pred = torch.argmax(logits, dim=1)
        ys.extend(y.detach().cpu().numpy().tolist())
        ps.extend(pred.detach().cpu().numpy().tolist())
    return float(accuracy_score(ys, ps)), float(f1_score(ys, ps, average="macro")), ys, ps

def freeze_backbone(model, model_name):
    if model_name == "resnet18":
        for n, p in model.named_parameters():
            if not n.startswith("fc."):
                p.requires_grad = False
    if model_name == "efficientnet_b0":
        for n, p in model.named_parameters():
            if not n.startswith("classifier."):
                p.requires_grad = False

def unfreeze_last_block(model, model_name):
    for p in model.parameters():
        p.requires_grad = True
    if model_name == "resnet18":
        for n, p in model.named_parameters():
            if not (n.startswith("layer4.") or n.startswith("fc.")):
                p.requires_grad = False
    if model_name == "efficientnet_b0":
        for n, p in model.named_parameters():
            if not ("features.7" in n or n.startswith("classifier.")):
                p.requires_grad = False

def train_image_experiment(
    ds,
    model_name,
    seed,
    epochs_head=2,
    epochs_ft=2,
    lr_head=3e-4,
    lr_ft=1e-4,
    batch_size=32,
    img_size=224,
    tfm_preset="baseline",
    use_class_weights=False,
    weight_decay=1e-2
):
    dl_tr, dl_va, dl_te, label_to_idx, idx_to_label, y_train_idx = make_image_loaders(
        ds, seed, batch_size=batch_size, img_size=img_size, tfm_preset=tfm_preset
    )
    num_classes = len(label_to_idx)

    model = build_model(model_name, num_classes).to(device)

    cw = None
    if use_class_weights:
        cw = compute_class_weights(y_train_idx, num_classes).to(device)

    loss_fn = nn.CrossEntropyLoss(weight=cw) if cw is not None else nn.CrossEntropyLoss()

    hist = []
    best = {"val_f1": -1, "state": None, "phase": None}

    freeze_backbone(model, model_name)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr_head, weight_decay=weight_decay)

    for ep in range(1, epochs_head + 1):
        model.train()
        losses = []
        for x, y in dl_tr:
            x = x.to(device)
            y = y.to(device) if torch.is_tensor(y) else torch.tensor(y).to(device)
            opt.zero_grad(set_to_none=True)
            logits = model(x)
            loss = loss_fn(logits, y)
            loss.backward()
            opt.step()
            losses.append(loss.item())
        va_acc, va_f1, _, _ = eval_loader(model, dl_va)
        rec = {"phase": "head", "epoch": ep, "train_loss": float(np.mean(losses)), "val_acc": va_acc, "val_f1_macro": va_f1}
        hist.append(rec)
        if va_f1 > best["val_f1"]:
            best["val_f1"] = va_f1
            best["state"] = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            best["phase"] = "head"

    if epochs_ft > 0:
        unfreeze_last_block(model, model_name)
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr_ft, weight_decay=weight_decay)

        for ep in range(1, epochs_ft + 1):
            model.train()
            losses = []
            for x, y in dl_tr:
                x = x.to(device)
                y = y.to(device) if torch.is_tensor(y) else torch.tensor(y).to(device)
                opt.zero_grad(set_to_none=True)
                logits = model(x)
                loss = loss_fn(logits, y)
                loss.backward()
                opt.step()
                losses.append(loss.item())
            va_acc, va_f1, _, _ = eval_loader(model, dl_va)
            rec = {"phase": "finetune", "epoch": ep, "train_loss": float(np.mean(losses)), "val_acc": va_acc, "val_f1_macro": va_f1}
            hist.append(rec)
            if va_f1 > best["val_f1"]:
                best["val_f1"] = va_f1
                best["state"] = {k: v.detach().cpu() for k, v in model.state_dict().items()}
                best["phase"] = "finetune"

    if best["state"] is not None:
        model.load_state_dict({k: v.to(device) for k, v in best["state"].items()})

    te_acc, te_f1, y_true, y_pred = eval_loader(model, dl_te)
    cm = confusion_matrix(y_true, y_pred)

    return {
        "dataset": ds,
        "model": model_name,
        "seed": seed,
        "img_size": img_size,
        "batch_size": batch_size,
        "tfm_preset": tfm_preset,
        "use_class_weights": use_class_weights,
        "epochs_head": epochs_head,
        "epochs_ft": epochs_ft,
        "lr_head": lr_head,
        "lr_ft": lr_ft,
        "weight_decay": weight_decay,
        "best_phase": best["phase"],
        "history": hist,
        "test": {"acc": float(te_acc), "f1_macro": float(te_f1)},
        "confusion_matrix": cm.tolist()
    }

3.5 Snimanje eksperimenata u Runs

Ovaj blok snima json i confusion_matrix.png po eksperimentu, tako da u Word-u imaš artefakte kao dokaz. Folder struktura je OUT_ROOT/dataset/model/seed_X/.

In [8]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

def save_confmat(cm, labels, title, out_path):
    fig, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=np.array(cm), display_labels=labels)
    disp.plot(ax=ax, values_format="d")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    plt.close(fig)

def save_image_experiment(res):
    ds = res["dataset"]
    model = res["model"]
    seed = res["seed"]
    meta = load_meta(ds)

    df_tr = load_split(ds, "train")
    ycol = label_col(meta, df_tr)
    labels = sorted(df_tr[ycol].astype(str).unique().tolist())

    exp_dir = OUT_ROOT / ds / model / f"seed_{seed}"
    exp_dir.mkdir(parents=True, exist_ok=True)

    save_json(exp_dir / "metrics.json", res)
    pd.DataFrame(res["history"]).to_csv(exp_dir / "history.csv", index=False)
    save_confmat(res["confusion_matrix"], labels, f"{ds} {model} seed={seed}", exp_dir / "confusion_matrix.png")

    return exp_dir

3.6 A: Robustnost (3 seeda) za sva tri image dataseta

Ovde merimo mean±std za baseline podešavanja, da možeš da kažeš da rezultat nije slučajan. Ovo je jeftino i brzo jer radimo samo head+kratak finetune.

In [9]:
image_datasets = ["lc25000", "sipakmed", "rm1000_lung_history"]
seeds = [42, 43, 44]

robust_results = []
for ds in image_datasets:
    for seed in seeds:
        res = train_image_experiment(
            ds=ds,
            model_name="resnet18",
            seed=seed,
            epochs_head=2,
            epochs_ft=1,
            lr_head=3e-4,
            lr_ft=1e-4,
            batch_size=32,
            img_size=224,
            tfm_preset="baseline",
            use_class_weights=False
        )
        save_image_experiment(res)
        robust_results.append(res)
        print(ds, seed, res["test"])

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 195MB/s]


lc25000 42 {'acc': 1.0, 'f1_macro': 1.0}
lc25000 43 {'acc': 1.0, 'f1_macro': 1.0}
lc25000 44 {'acc': 1.0, 'f1_macro': 1.0}
sipakmed 42 {'acc': 0.9636963696369637, 'f1_macro': 0.9637060911659642}
sipakmed 43 {'acc': 0.9389438943894389, 'f1_macro': 0.9388314141826596}
sipakmed 44 {'acc': 0.9554455445544554, 'f1_macro': 0.9558057656567446}
rm1000_lung_history 42 {'acc': 0.9951111111111111, 'f1_macro': 0.9951111089382706}
rm1000_lung_history 43 {'acc': 0.9964444444444445, 'f1_macro': 0.9964443875546453}
rm1000_lung_history 44 {'acc': 0.992, 'f1_macro': 0.9920038128894649}


3.6.1 Rezime robustnosti (mean±std)

Ovo pravi tabelu za Word i Excel, sa prosekom i standardnom devijacijom. Ako je std mali, rezultat je stabilan.

In [10]:
rows = []
for r in robust_results:
    rows.append({
        "dataset": r["dataset"],
        "model": r["model"],
        "seed": r["seed"],
        "test_acc": r["test"]["acc"],
        "test_f1_macro": r["test"]["f1_macro"]
    })

df_rob = pd.DataFrame(rows)
summary_rob = df_rob.groupby(["dataset", "model"]).agg(
    acc_mean=("test_acc", "mean"),
    acc_std=("test_acc", "std"),
    f1_mean=("test_f1_macro", "mean"),
    f1_std=("test_f1_macro", "std"),
).reset_index()

summary_rob.to_csv(OUT_ROOT / "robustness_summary.csv", index=False)
summary_rob

,dataset,model,acc_mean,acc_std,f1_mean,f1_std
0,lc25000,resnet18,1.000000,0.000000,1.000000,0.000000
1,rm1000_lung_history,resnet18,0.994519,0.002281,0.994520,0.002279
2,sipakmed,resnet18,0.952695,0.012603,0.952781,0.012710


3.7 B: SipakMed tuning (augment + weights + model swap)

Ovde ciljamo da podignemo macro F1 na SipakMed, jer tu ima smisla. Testiramo mali grid i uzmemo najbolji.

In [11]:
sipak_grid = [
    {"model": "resnet18", "tfm": "baseline", "cw": False},
    {"model": "resnet18", "tfm": "strong",   "cw": False},
    {"model": "resnet18", "tfm": "strong",   "cw": True},
    {"model": "efficientnet_b0", "tfm": "baseline", "cw": False},
    {"model": "efficientnet_b0", "tfm": "strong",   "cw": True},
]

sipak_results = []
for cfg in sipak_grid:
    res = train_image_experiment(
        ds="sipakmed",
        model_name=cfg["model"],
        seed=42,
        epochs_head=3,
        epochs_ft=2,
        lr_head=3e-4,
        lr_ft=1e-4,
        batch_size=32,
        img_size=224,
        tfm_preset=cfg["tfm"],
        use_class_weights=cfg["cw"]
    )
    d = save_image_experiment(res)
    sipak_results.append(res)
    print("Saved:", d, "Test:", res["test"], "cfg:", cfg)

df_sipak = pd.DataFrame([{
    "model": r["model"],
    "tfm": r["tfm_preset"],
    "class_weights": r["use_class_weights"],
    "test_acc": r["test"]["acc"],
    "test_f1_macro": r["test"]["f1_macro"]
} for r in sipak_results]).sort_values("test_f1_macro", ascending=False)

df_sipak.to_csv(OUT_ROOT / "sipakmed_tuning.csv", index=False)
df_sipak

Saved: /content/drive/MyDrive/Diplomski/Runs/experiments_20260312_092942/sipakmed/resnet18/seed_42 Test: {'acc': 0.9620462046204621, 'f1_macro': 0.9619297070659139} cfg: {'model': 'resnet18', 'tfm': 'baseline', 'cw': False}
Saved: /content/drive/MyDrive/Diplomski/Runs/experiments_20260312_092942/sipakmed/resnet18/seed_42 Test: {'acc': 0.9504950495049505, 'f1_macro': 0.9507121369761247} cfg: {'model': 'resnet18', 'tfm': 'strong', 'cw': False}
Saved: /content/drive/MyDrive/Diplomski/Runs/experiments_20260312_092942/sipakmed/resnet18/seed_42 Test: {'acc': 0.9554455445544554, 'f1_macro': 0.9556188746860131} cfg: {'model': 'resnet18', 'tfm': 'strong', 'cw': True}
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 157MB/s]


Saved: /content/drive/MyDrive/Diplomski/Runs/experiments_20260312_092942/sipakmed/efficientnet_b0/seed_42 Test: {'acc': 0.905940594059406, 'f1_macro': 0.9056250119334945} cfg: {'model': 'efficientnet_b0', 'tfm': 'baseline', 'cw': False}
Saved: /content/drive/MyDrive/Diplomski/Runs/experiments_20260312_092942/sipakmed/efficientnet_b0/seed_42 Test: {'acc': 0.8943894389438944, 'f1_macro': 0.8941366984744461} cfg: {'model': 'efficientnet_b0', 'tfm': 'strong', 'cw': True}


,model,tfm,class_weights,test_acc,test_f1_macro
0,resnet18,baseline,False,0.962046,0.961930
2,resnet18,strong,True,0.955446,0.955619
1,resnet18,strong,False,0.950495,0.950712
3,efficientnet_b0,baseline,False,0.905941,0.905625
4,efficientnet_b0,strong,True,0.894389,0.894137


3.8 Thyroid k-fold (stabilnost na tabularu)

Ovde proveravamo stabilnost rezultata na malom tabularnom skupu kroz 5-fold. Dobijaš mean±std za accuracy i macro F1, što je baš dobro za diplomski.

In [12]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

def thyroid_kfold(seed=42, n_splits=5):
    ds = "thyroid_recurrence"
    meta = load_meta(ds)
    df = load_split(ds, "train")
    ycol = label_col(meta, df)
    X = df.drop(columns=[ycol])
    y = df[ycol]

    num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    cat_cols = [c for c in X.columns if c not in num_cols]

    pre = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
            ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
        ]
    )

    models = {
        "LogReg": Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000))]),
        "RandomForest": Pipeline([("pre", pre), ("clf", RandomForestClassifier(n_estimators=400, random_state=seed, n_jobs=-1))])
    }

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    out = []

    for name, model in models.items():
        accs, f1s = [], []
        for tr_idx, va_idx in skf.split(X, y):
            X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
            y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
            model.fit(X_tr, y_tr)
            p = model.predict(X_va)
            accs.append(accuracy_score(y_va, p))
            f1s.append(f1_score(y_va, p, average="macro"))
        out.append({
            "model": name,
            "acc_mean": float(np.mean(accs)),
            "acc_std": float(np.std(accs, ddof=1)),
            "f1_mean": float(np.mean(f1s)),
            "f1_std": float(np.std(f1s, ddof=1))
        })

    return pd.DataFrame(out)

df_kfold = thyroid_kfold(seed=42, n_splits=5)
df_kfold.to_csv(OUT_ROOT / "thyroid_kfold.csv", index=False)
df_kfold

,model,acc_mean,acc_std,f1_mean,f1_std
0,LogReg,0.956863,0.037716,0.946282,0.048526
1,RandomForest,0.956863,0.032219,0.946766,0.040335


3.9 Završni “master summary”

Ovo pravi jedan CSV sa svim ključnim rezultatima (robustnost + sipak tuning + thyroid kfold). Taj fajl je idealan da ga kasnije samo ubaciš u Word i Excel.

In [13]:
master = []

for _, r in summary_rob.iterrows():
    master.append({
        "group": "robustness_3seeds",
        "dataset": r["dataset"],
        "model": r["model"],
        "acc_mean": r["acc_mean"],
        "acc_std": r["acc_std"],
        "f1_mean": r["f1_mean"],
        "f1_std": r["f1_std"],
    })

best_sipak = df_sipak.iloc[0].to_dict() if len(df_sipak) else {}
if best_sipak:
    master.append({
        "group": "sipakmed_best",
        "dataset": "sipakmed",
        "model": best_sipak["model"],
        "acc_mean": best_sipak["test_acc"],
        "acc_std": 0.0,
        "f1_mean": best_sipak["test_f1_macro"],
        "f1_std": 0.0,
    })

for _, r in df_kfold.iterrows():
    master.append({
        "group": "thyroid_kfold",
        "dataset": "thyroid_recurrence",
        "model": r["model"],
        "acc_mean": r["acc_mean"],
        "acc_std": r["acc_std"],
        "f1_mean": r["f1_mean"],
        "f1_std": r["f1_std"],
    })

df_master = pd.DataFrame(master)
df_master.to_csv(OUT_ROOT / "master_summary.csv", index=False)
df_master

,group,dataset,model,acc_mean,acc_std,f1_mean,f1_std
0,robustness_3seeds,lc25000,resnet18,1.000000,0.000000,1.000000,0.000000
1,robustness_3seeds,rm1000_lung_history,resnet18,0.994519,0.002281,0.994520,0.002279
2,robustness_3seeds,sipakmed,resnet18,0.952695,0.012603,0.952781,0.012710
3,sipakmed_best,sipakmed,resnet18,0.962046,0.000000,0.961930,0.000000
4,thyroid_kfold,thyroid_recurrence,LogReg,0.956863,0.037716,0.946282,0.048526
5,thyroid_kfold,thyroid_recurrence,RandomForest,0.956863,0.032219,0.946766,0.040335
